<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>Générer Excel Shap</h1>
</div>

和原来的那个区别在于，这个不需要生成ptf，而是直接结合screen和shap

In [1]:
def add_icb_supersector_names(dataframe, icb_code_column=' Benchmark ICB Supersector '):
    """
    Add ICB Supersector names to a dataframe based on ICB code numbers.
    
    Parameters:
    -----------
    dataframe : pandas.DataFrame
        The dataframe containing ICB supersector codes
    icb_code_column : str, default=' Benchmark ICB Supersector '
        The name of the column containing ICB supersector codes
        
    Returns:
    --------
    pandas.DataFrame
        The dataframe with a new 'Supersector' column containing the ICB supersector names
    """
    # ICB Supersector mapping (name to code)
    icb_supersectors = {  
        "Auto & Parts": 1,  
        "Banks": 2,  
        "Basic Resources": 3,  
        "Chemicals": 4,  
        "Construction": 5,  
        "Financial Services": 6,  
        "Food, Beverage & Tobacco": 7,  
        "Health Care": 8,  
        "Industrial Goods & Services": 9,  
        "Insurance": 10,  
        "Media": 11,  
        "Energy": 12,  
        "Personal & Household Goods": 13,  
        "Real Estate": 14,  
        "Retail": 15,  
        "Technology": 16,  
        "Telecommunications": 17,  
        "Travel & Leisure": 18,  
        "Utilities": 19  
    }

    # Create a reverse mapping dictionary (code -> name)  
    icb_supersectors_reverse = {v: k for k, v in icb_supersectors.items()}  

    # Add a new column with the supersector name  
    dataframe_updated = dataframe.copy()
    dataframe_updated['Supersector'] = dataframe_updated[icb_code_column].map(icb_supersectors_reverse)
    
    return dataframe_updated

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os 
from pathlib import Path
import sys
import pandas as pd
import numpy as np
try:
    from xbbg import blp
except ImportError:
    blp = None
from datetime import datetime, timedelta
from Codes.ML_SUIVI import *
from Config import config_EU, config_US
from Codes.BacktestEngine import merge_ticker_secondaire
%load_ext autoreload
%autoreload 2

_TP_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tp_core").exists()), Path(r"C:\GoogleDrive\TP"))
if str(_TP_ROOT) not in sys.path:
    sys.path.insert(0, str(_TP_ROOT))
from tp_core.data_sources import RETURNS_PATH as CANONICAL_RETURNS_PATH
from tp_core.data_sources import SCREEN_AGGREGATE_PATH

list_noire_path = r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_BASE\_ ESG DATA\Liste_Noire_Exclusion.xlsx"
path_PTF = r"Portfolio_BT\PTF_ESG_US.parquet"
screen_path = str(SCREEN_AGGREGATE_PATH)
returns_path = str(CANONICAL_RETURNS_PATH)
# path_ciq = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\last_screenCIQ.parquet"
mail_output_path =  r"SUIVI\\"

shap_grouped = combine_score_shap(config_EU, config_US)

# Ajuster les outliers
shap_grouped = process_outlier_rows_only(shap_grouped, columns=['Dividend Avg Percentile', 'Value Avg Percentile',
        'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile',
        'Growth Avg Percentile', 'Value Avg Percentile_change_1M',
        'Quality Avg Percentile_change_1M', 'Growth Avg Percentile_change_1M',
        'Value Avg Percentile_change_3M', 'Quality Avg Percentile_change_3M',
        'Growth Avg Percentile_change_3M', 'Value Avg Percentile_change_6M',
        'Quality Avg Percentile_change_6M', 'Growth Avg Percentile_change_6M',
        'Value Avg Percentile_change_12M', 'Quality Avg Percentile_change_12M',
        'Growth Avg Percentile_change_12M', 'Sector 1', 'Sector 2', 'Sector 3',
        'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
        'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
        'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19'], sum_col="average_prediction")

last_date = shap_grouped['Date'].max()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
All max dates are identical


In [4]:
screen = pd.read_parquet(screen_path)
date = closest_date_in_df(last_date, screen)
screen = screen[screen['Date'] == date]
screen = screen[(screen[f"Weight in SP500"] > 0) | (screen[f"Weight in STOXX EUROPE 600"] > 0)]
screen = add_icb_supersector_names(screen)

In [5]:
screen_display_columns = [
    "Weight in SP500", "Weight in STOXX EUROPE 600", "Name", "Supersector",
    "Exchange Country Region", "Exchange Country Name",
    "Reco Analyst",
    "PE LTM", "EPS Growth FY1", "ROE avg FY0",
    "Oper Margin",
    "DVD Yield FY0", "Earns Yield FY0",
]
missing_screen_display_columns = [col for col in screen_display_columns if col not in screen.columns]
if missing_screen_display_columns:
    print("Colonnes screen absentes, remplies avec NaN:", missing_screen_display_columns)
    for col in missing_screen_display_columns:
        screen[col] = np.nan

df_screen = screen[screen_display_columns].copy()

In [6]:
df_total = df_screen.merge(shap_grouped,                                 
                        how="left",
                        left_on="ISIN",
                        right_on="ISIN"
                        )
# df_total = add_raison_repechage(df_total, path_PTF)

# Aggregate les Impact Facto et Secto
df_total['Contrib Facto'] = df_total[['Dividend Avg Percentile', 'Value Avg Percentile',
                                'Quality Avg Percentile', 'Mom Avg Percentile', 'LowVol Avg Percentile',
                                'Growth Avg Percentile', 'Value Avg Percentile_change_1M',
                                'Quality Avg Percentile_change_1M', 'Growth Avg Percentile_change_1M',
                                'Value Avg Percentile_change_3M', 'Quality Avg Percentile_change_3M',
                                'Growth Avg Percentile_change_3M', 'Value Avg Percentile_change_6M',
                                'Quality Avg Percentile_change_6M', 'Growth Avg Percentile_change_6M',
                                'Value Avg Percentile_change_12M', 'Quality Avg Percentile_change_12M',
                                'Growth Avg Percentile_change_12M']].sum(axis=1)

df_total['Contrib Secto'] = df_total[['Sector 1', 'Sector 2', 'Sector 3',
                                'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
                                'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
                                'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19']].sum(axis=1)
df_total = rename_cols_1(df_total)  # Change specific columns' name
df_total = rename_cols_2(df_total)  # Change contribution related columns' name
screen_original_data = get_factos()
screen_original_data = rename_cols_3(screen_original_data)  # Change original score columns' name

df_total = df_total.merge(screen_original_data,                             
                how="left",
                left_on="ISIN",
                right_on="ISIN"
                )

# df_total = add_esg_list_noire(list_noire_path, df_total)
list_exclu_total = get_list_exclusion()
df_total = df_total.merge(list_exclu_total[['ISIN', 'Raison Exclusion']], how="left", on='ISIN')

# Cols to show in Excel
list_cols_excel = [
                'Date','ISIN', 'Company SEDOL', 'Name', "Region", 'Country', "Weight in SP500", "Weight in STOXX EUROPE 600",
                'Supersector',  
                'DVD Yield', 'Earnings Yield', 'ROE', 'PE', "Oper Margin",
                'Score ML', 'Predicted Forward Return 1M',

                'Contrib Facto',
                'Value Contrib', 'Value Change_1M', 'Value Change_3M', 'Value Change_6M', 'Value Change_12M', 
                'Value Score', 'Value Score Change_1M', 'Value Score Change_3M', 'Value Score Change_6M', 'Value Score Change_12M', 

                'Quality Contrib', 'Quality Change_1M', 'Quality Change_3M', 'Quality Change_6M', 'Quality Change_12M',
                'Quality Score', 'Quality Score Change_1M', 'Quality Score Change_3M', 'Quality Score Change_6M', 'Quality Score Change_12M',

                'Growth Contrib', 'Growth Change_1M', 'Growth Change_3M', 'Growth Change_6M', 'Growth Change_12M', 
                'Growth Score', 'Growth Score Change_1M', 'Growth Score Change_3M', 'Growth Score Change_6M', 'Growth Score Change_12M', 

                'Dividend Contrib', 'Mom Contrib', 'LowVol Contrib',
                'Dividend Score', 'Mom Score', 'LowVol Score',
                
                'Contrib Secto',
                'Sector 1', 'Sector 2', 'Sector 3',
                'Sector 4', 'Sector 5', 'Sector 6', 'Sector 7', 'Sector 8', 'Sector 9',
                'Sector 10', 'Sector 11', 'Sector 12', 'Sector 13', 'Sector 14',
                'Sector 15', 'Sector 16', 'Sector 17', 'Sector 18', 'Sector 19'
                ]

basket = df_total[list_cols_excel]

basket['Score ML'] = basket.groupby(['Region', 'Date', 'Supersector'])['Score ML'].rank(  
                                                        pct=True,  
                                                        ascending=True  
                                                        ) * 10
# ==============================================================================
# 分组权重归一化 (Rebalancing within groups)
# ==============================================================================
# 定义分组键
group_cols = ['Region', 'Date', 'Supersector']

# 定义需要处理的权重列 (根据你的描述是这两个)
weight_cols = ["Weight in SP500", "Weight in STOXX EUROPE 600"]

for col in weight_cols:
    # 检查列是否存在于 dataframe 中，避免报错
    if col in basket.columns:
        # 1. 计算每个分组内的权重总和
        group_sums = basket.groupby(group_cols)[col].transform('sum')
        
        # 2. 执行归一化：权重 / 组内总和
        # 使用 numpy 的 where 避免除以 0 产生 inf/nan
        # 如果组内总和不为0，则除以总和；如果为0，则保持原值（通常是0）
        import numpy as np
        basket[col] = np.where(group_sums != 0, basket[col] / group_sums, 0.0)

        # 3. (可选) 填充可能产生的 NaN 为 0
        basket[col] = basket[col].fillna(0.0)
        
        print(f"Rebalanced '{col}' within groups {group_cols}.")
# ==============================================================================

basket_et_univ = basket.copy(deep=True)
basket_et_univ.to_excel(os.path.join(mail_output_path, r"Explicativity.xlsx"), index=False)

Rebalanced 'Weight in SP500' within groups ['Region', 'Date', 'Supersector'].
Rebalanced 'Weight in STOXX EUROPE 600' within groups ['Region', 'Date', 'Supersector'].
